# What Billy Penn Can No Longer See

The slice of horizon each tower above his hat blocks, from the top of City Hall. Run `fetch_week1.py day5` first.

## 1 · Settings

In [ ]:
DAY      = 5
PROMPT   = "Sight"
SLUG     = "sight-billy-penn"
NOTEBOOK = "day05_sight_billy_penn.ipynb"
TITLE    = "What Billy Penn\nCan No Longer See"
CREDITS  = "Footprints: City of Philadelphia. Tower heights: Wikipedia, List of tallest buildings in Philadelphia.\nStreets and water: © OpenStreetMap contributors."

## 2 · Load

In [ ]:
import geopandas as gpd, numpy as np, pandas as pd
from shapely.geometry import Polygon
from mapkit import poster, publish
from mapkit.paths import PROCESSED

footprints = gpd.read_file(PROCESSED / "philly_tall.geojson")
streets = gpd.read_file(PROCESSED / "philly_streets.geojson")
water = gpd.read_file(PROCESSED / "philly_water.geojson")

## 3 · Analysis, every number on the map comes from here

In [ ]:
HAT_FT = 548                                 # top of the statue, City Hall's listed height
HAT = (-75.16353, 39.95274)                  # the City Hall tower, from OSM's 167 m tower footprint
TREATY_PARK = (-75.1292, 39.9666)            # Penn Treaty Park, where the statue faces

# Every tower with a listed height above the hat, matched to its city footprint by address
TOWERS = {"1800 ARCH ST": ("Comcast Technology Center", 1113), "1701 JOHN F KENNEDY BLVD # 1": ("Comcast Center", 973),
          "1650 MARKET ST": ("One Liberty Place", 945), "1601 CHESTNUT ST": ("Two Liberty Place", 848),
          "1735 MARKET ST": ("BNY Mellon Center", 792), "1717 ARCH ST": ("Three Logan Square", 739),
          "2929 WALNUT ST": ("FMC Tower", 730), "1901-17 MARKET ST": ("G. Fred DiBona Jr. Building", 625),
          "1441 CHESTNUT ST": ("The W and Element", 617), "1911 WALNUT ST": ("The Laurel", 599),
          "2005 MARKET ST": ("One Commerce Square", 572), "2001 MARKET ST": ("Two Commerce Square", 572)}
towers = footprints[footprints.address.isin(TOWERS)].copy()
towers["tower"] = towers.address.map(lambda a: TOWERS[a][0])
towers["listed_ft"] = towers.address.map(lambda a: TOWERS[a][1])
assert len(towers) == len(TOWERS), "a tower is missing a footprint"

m = towers.to_crs(32618)
hat_x, hat_y = gpd.GeoSeries(gpd.points_from_xy([HAT[0]], [HAT[1]]), crs=4326).to_crs(32618).iloc[0].coords[0]

def bearing(x, y):
    """Compass bearing in degrees from the hat."""
    return (np.degrees(np.arctan2(x - hat_x, y - hat_y))) % 360

def span(geom):
    """The slice of horizon a footprint blocks, measured from the hat, plus its nearest distance."""
    parts = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    xy = np.vstack([np.array(p.exterior.coords) for p in parts])
    b = bearing(xy[:, 0], xy[:, 1])
    middle = bearing(*geom.centroid.coords[0])
    offset = ((b - middle + 180) % 360) - 180
    return middle + offset.min(), middle + offset.max(), float(np.hypot(*(xy - [hat_x, hat_y]).T).min()), middle

rows = [span(g) for g in m.geometry]
m[["from_deg", "to_deg", "near_m", "mid_deg"]] = rows

# Add up the horizon without double counting towers that overlap
degrees = np.zeros(3600, bool)
for a, b in zip(m.from_deg, m.to_deg):
    degrees[(np.arange(int(a * 10), int(np.ceil(b * 10))) % 3600)] = True
blocked_pct = degrees.mean() * 100
tp = gpd.GeoSeries(gpd.points_from_xy([TREATY_PARK[0]], [TREATY_PARK[1]]), crs=4326).to_crs(32618).iloc[0]
gaze = float(bearing(tp.x, tp.y))
gaze_blocked = bool(degrees[int(gaze * 10) % 3600])
nearest_to_gaze = float(min(min(abs((a - gaze + 180) % 360 - 180), abs((b - gaze + 180) % 360 - 180)) for a, b in zip(m.from_deg, m.to_deg)))
print(f"Towers above the hat   {len(m)}")
print(f"Horizon blocked        {blocked_pct:.1f}% of 360 degrees")
print(f"Blocked bearings       {m.from_deg.min()%360:.0f} to {m.to_deg.max()%360:.0f}")
print(f"Bearing to Treaty Park {gaze:.0f} degrees, blocked: {gaze_blocked}, closest tower edge {nearest_to_gaze:.0f} degrees away")
print(m[["tower", "listed_ft", "mid_deg", "near_m"]].sort_values("mid_deg").round(0).to_string(index=False))

## 4 · Map

In [ ]:
fig, ax = poster.page("#0b1115")
R = 15000
def wedge(a, b, near, far=R):
    t = np.radians(np.linspace(a, b, 24))
    outer = np.c_[hat_x + far * np.sin(t), hat_y + far * np.cos(t)]
    inner = np.c_[hat_x + near * np.sin(t[::-1]), hat_y + near * np.cos(t[::-1])]
    return Polygon(np.vstack([outer, inner]))

wedges = gpd.GeoSeries([wedge(a, b, n) for a, b, n in zip(m.from_deg, m.to_deg, m.near_m)], crs=32618).to_crs(3857)
hx, hy = poster.merc(*HAT)
half_w = 5200
half_h = half_w / 0.8
shift_x, shift_y = -half_w * 0.17, -half_h * 0.10
ax.set_xlim(hx + shift_x - half_w, hx + shift_x + half_w)
ax.set_ylim(hy + shift_y - half_h, hy + shift_y + half_h)
xlim, ylim = ax.get_xlim(), ax.get_ylim()

water.to_crs(3857).plot(ax=ax, color="#16242c", lw=0, zorder=1)
s = streets.to_crs(3857).cx[xlim[0]:xlim[1], ylim[0]:ylim[1]]
s.plot(ax=ax, color="#4a5b63", lw=0.34, zorder=2)
wedges.plot(ax=ax, color=poster.PINK, alpha=0.26, lw=0, zorder=3)
wedges.boundary.plot(ax=ax, color=poster.PINK, alpha=0.55, lw=0.4, zorder=3)
m.to_crs(3857).plot(ax=ax, color=poster.PINK, lw=0, zorder=5)

# where he's actually looking
tx, ty = poster.merc(*TREATY_PARK)
ax.plot([hx, tx], [hy, ty], color="white", lw=1.2, ls=(0, (1, 2.4)), zorder=6)
ax.scatter([tx], [ty], s=40, color="white", zorder=7, lw=0)
poster.halo_text(ax, tx - 60, ty + 170, "Penn Treaty Park,\nwhere he's facing", size=13.5, path=poster.LATO_BOLD, ha="right", va="bottom", zorder=8, halo="#0b1115", linespacing=1.25)
ax.scatter([hx], [hy], s=46, color="white", zorder=8, lw=0)
poster.halo_text(ax, hx + 130, hy - 130, "Billy Penn, 548 ft", size=13.5, path=poster.LATO_BOLD, va="top", zorder=8, halo="#0b1115")
ax.set_xlim(xlim); ax.set_ylim(ylim)
poster.title(fig, TITLE, size=38, x=0.06, y=0.955, path=poster.CINZEL, sub_path=poster.LATO,
             subtitle=f"The William Penn statue tops Philadelphia's City Hall, 548 ft up. {len(m)} towers now stand taller.\nEach pink wedge is the view one of them blocks, {blocked_pct:.0f}% of his horizon in all. He faces the other way.", sub_size=14, sub_color="#d3dde0")
poster.credit(fig, CREDITS, path=poster.LATO, name_path=poster.LATO_BOLD, color="#a9b6ba", name_color="#eef3f4")

## 5 · Export

In [ ]:
publish.save(fig, day=DAY, slug=SLUG, prompt=PROMPT, title=TITLE.replace("\n", " "),
             subtitle=f"{blocked_pct:.0f}% of his horizon is gone", series="city", tool="Python",
             notebook=NOTEBOOK, credits=CREDITS)